# Notebook 8 – Transfer Learning Basics



In [1]:
import time
import numpy as np
import torch
from torch import nn
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

PyTorch: 2.14.0+cpu | device: cpu


## Concept and explanation
A pretrained backbone has reusable visual features. Feature extraction freezes it and trains a new classifier; fine-tuning unfreezes some layers and adapts them using a smaller learning rate. Early layers often represent generic edges/textures; later features are more task-specific. Transfer tends to work best when source and target image domains are related.

In [2]:
from torchvision.models import resnet18,ResNet18_Weights
model=resnet18(weights=None) # use ResNet18_Weights.DEFAULT for actual pretrained weights
for p in model.parameters(): p.requires_grad=False
features=model.fc.in_features; model.fc=nn.Linear(features,5)
print('head-only trainable parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad))
for p in model.layer4.parameters(): p.requires_grad=True
optimizer=torch.optim.Adam([{'params':model.fc.parameters(),'lr':1e-3},{'params':model.layer4.parameters(),'lr':1e-4}])
print('after unfreezing layer4:',sum(p.numel() for p in model.parameters() if p.requires_grad))

head-only trainable parameters: 2565
after unfreezing layer4: 8396293


## Implement and interpret
Flow: pretrained model → freeze backbone → replace classifier → train the head. Then pretrained model → unfreeze selected layers → fine-tune. `weights=None` keeps this mechanics demo offline; `ResNet18_Weights.DEFAULT` downloads weights and supplies a compatible preprocessing transform.

ImageNet-to-everyday objects is a related-domain transfer. Medical, satellite, microscopy, and industrial images may mismatch in color, texture, scale, or acquisition. Compare frozen validation results with fine-tuning and a scratch baseline. Start frozen; unfreeze further only when validation evidence suggests the representation is limiting. Consider BatchNorm running statistics on tiny target datasets.